In [1]:
import os
from dotenv import load_dotenv
from scraper import fetch_website_links,fetch_Website_contents
from IPython.display import Markdown, display, update_display
from openai import OpenAI
from groq import Groq
MODEL = "openai/gpt-oss-120b:free"


In [ ]:
import os
api_key = os.environ.get("AI_API_KEY")
if api_key:
    print("API KEY IS VALID")
else:
    raise ValueError("API key is not valid and set")

openai = OpenAI(
    api_key=api_key,
    base_url="https://openrouter.ai/api/v1"
)

In [6]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are expert at deciding which links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs page.
You should respond in JSON as in this example:

{
    "links" : [
        {"type":"about page","url":"https://full.url/goes/here/about" },
        {"type":"careers page","url":"https://full.url/goes/here/careers" }
    ]
}
"""

In [23]:
def get_links_user_propmt(url):
    user_prompt = f"""
    Here is the list of the links on the website {url} -
    Please decide which of these are relevant web links for a brochure about the company,
    respond with the full https URL in JSON format.
    Do not include Terms of Service,Privacy , email links.

    Here are the links (Note that some links might me relative):
 """
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [24]:
import json

def select_relevant_links(url):
    print(f"Selecting relevant links for the url : {url} by the model : {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role":"system","content":link_system_prompt},
            {"role":"user","content":get_links_user_propmt(url)}

        ],
        response_format={"type":"json_object"}
    )
    result = response.choices[0].message.content
    print(f"API Response: {result}")  # Debug: print actual response
    
    try:
        links = json.loads(result)
    except json.JSONDecodeError as e:
        print(f"JSON Decode Error: {e}")
        print(f"Response content: '{result}'")
        # Return a default structure if parsing fails
        links = {"links": []}
    
    print(f"Found {len(links.get('links', []))} relevant links")
    return links


In [8]:
print(select_relevant_links("https://huggingface.co"))

Selecting relevant links for the url : https://huggingface.co by the model : openai/gpt-oss-120b:free
Found 1 relevant links
{'links': [{'type': 'about page', 'url': 'https://huggingface.co/brand'}, {'type': 'about page', 'url': 'https://huggingface.co/learn'}, {'type': 'company page', 'url': 'https://huggingface.co/enterprise'}, {'type': 'blog page', 'url': 'https://huggingface.co/blog'}, {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'}, {'type': 'careers page', 'url': 'https://huggingface.co/join'}]}


In [33]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_Website_contents(url)
    relevant_links = select_relevant_links(url)
    # print(relevant_links)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    
    links_list = relevant_links.get('links', [])
    if not links_list:
        result += "\nNo relevant links found.\n"
    else:
        for link in links_list:
            result += f"\n\n### Link: {link['type']}\n"
            result += fetch_Website_contents(link["url"])
    
    return result


In [ ]:
print(fetch_page_and_all_relevant_links("https://huggingface.co/"))

In [26]:
# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short brochure about the company for prespective customers, investors and recruits
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company in a humorous, sarcastic and roasting style as by a Roasting expert.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

In [27]:
def get_brochure_user_prompt(company_name,url):
    user_prompt = f""" 
You are lopkoing at a company called : {company_name}
Here are the contents of its landing page and other relevant pages:
use this information to build a short brochure of the company in markdown format without code blocks.\n\n
"""
    
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000]
    return user_prompt

In [ ]:
print(get_brochure_user_prompt("Hugging face","https://huggingface.co"))

In [ ]:
def create_brochure(company_name,url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role":"system","content":brochure_system_prompt},
            {"role":"user","content":get_brochure_user_prompt(company_name,url)}
        ]
    )
    result = response.choices[0].message.content


In [30]:
def stream_brochure(company_name,url):
    stream= openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role":"system","content":brochure_system_prompt},
            {"role":"user","content":get_brochure_user_prompt(company_name,url)}
        ],
        stream=True
    )
    result = ""
    display_handle = display(Markdown(""),display_id=True)
    for chunk in stream:
        result += chunk.choices[0].delta.content or ''
        update_display(Markdown(result),display_id=display_handle.display_id)

In [35]:
brochure_system_prompt_2 = """
You are an assistant who is expert at Kannada language literature and translate the provided content to kannada word to word accurately.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""
def create_brochure(company_name,url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role":"system","content":brochure_system_prompt},
            {"role":"user","content":get_brochure_user_prompt(company_name,url)}
        ]
    )
    result = response.choices[0].message.content
    return result
user_prompt_2 = f"""
Here the contents of a company's brochure that need to be translated into kannada language
Contents :
"""

def stream_brochure(company_name,url):
    stream= openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role":"system","content":brochure_system_prompt_2 },
            {"role":"user","content":user_prompt_2+create_brochure(company_name,url)}
        ],
        stream=True
    )
    result = ""
    display_handle = display(Markdown(""),display_id=True)
    for chunk in stream:
        result += chunk.choices[0].delta.content or ''
        update_display(Markdown(result),display_id=display_handle.display_id)

In [7]:
gpt_model = "openai/gpt-oss-120b"
gemma_model = "openai/gpt-oss-20b"

gpt_prompt = "You are a chatbot who is argumentative and aggressive\
    You disagree with anything in the conservation and you challenge everything, in a snarky way.Make sure you alway provide short replies" 

gemma_prompt = "You are a very polite, courteous chatbot, You try to agree with\
    everything the other person says, or find common ground. If the other person is argumentative ,\
        you to try calm them down and keep chatting.Make sure to chat as short as possible."

gpt_msgs = ["Hi there!"]
gemma_msgs = ["Hey"]

In [8]:
def call_gpt():
    messages = [{"role":"system","content":gpt_prompt}]
    for gpt,gemma in zip(gpt_msgs,gemma_msgs):
        messages.append({"role":"assistant","content":gpt})
        messages.append({"role":"user","content":gemma})
    response = openai.chat.completions.create(
        model=gpt_model,
        messages=messages
    )
    return response.choices[0].message.content

In [10]:
print(call_gpt())

AuthenticationError: Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}

In [5]:
def call_gemma():
    messages = [{"role":"system","content":gemma_prompt}]
    for gpt,gemma in zip(gpt_msgs,gemma_msgs):
        messages.append({"role":"assistant","content":gemma})
        messages.append({"role":"user","content":gpt})
    messages.append({"role":"user","content":gpt_msgs[-1]})
    response = openai.chat.completions.create(
        model=gemma_model,
        messages=messages
    )
    return response.choices[0].message.content

In [27]:
print(call_gemma())

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '50', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1776211200000'}, 'provider_name': None}}, 'user_id': 'user_3C0sv1mPN99JdAjSAKjV0iuDQAv'}

In [6]:
import time
for i in range(5):
    gpt_res = call_gpt()
    display(Markdown(f"### GPT :{gpt_res}\n"))
    gpt_msgs.append(gpt_res)

    time.sleep(30)
    gemma_res = call_gemma()
    display(Markdown(f"### GEMMA :{gemma_res}\n"))
    gemma_msgs.append(gemma_res)


### GPT :Oh, great, another conversation starter. What do you want this time?


RateLimitError: Error code: 429 - {'error': {'message': 'Provider returned error', 'code': 429, 'metadata': {'raw': 'google/gemma-4-31b-it:free is temporarily rate-limited upstream. Please retry shortly, or add your own key to accumulate your rate limits: https://openrouter.ai/settings/integrations', 'provider_name': 'Google AI Studio', 'is_byok': False}}, 'user_id': 'user_3C0sv1mPN99JdAjSAKjV0iuDQAv'}